# MangaTR: ترجمة فصل كامل

الفصل كله: كشف ← OCR ← تنظيف ← ترجمة ← كتابة ← تصدير. بيطلّع `chapter.zip` (أو cbz/pdf) و`report.json` (لكل صفحة: عدد النصوص، المترجم، الأخطاء، الأزمنة).

**قبل ما تبدأ:** runtime **T4 GPU**، وأسرار Colab: `GITHUB_TOKEN` (للريبو الخاص) ومفتاح الترجمة، واحد على الأقل من `GEMINI_API_KEY` أو `OPENAI_API_KEY` أو `ANTHROPIC_API_KEY` (ممكن كذا مفتاح مفصولين بفاصلة للتدوير). المفاتيح ما بتتطبعش ولا بتتكتب في ملف.

In [ ]:
# 1) الـ GPU
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "مفيش GPU")

In [ ]:
# 2) تحميل الكود وتثبيته
import os
import subprocess

from google.colab import userdata

BRANCH = "phase1/core-logic"
REPO_DIR = "/content/MangaTR"
PUBLIC_URL = "https://github.com/Nader-Geek/MangaTR.git"

if not os.path.isdir(REPO_DIR):
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = (
        PUBLIC_URL.replace("https://", f"https://x-access-token:{token}@") if token else PUBLIC_URL
    )
    done = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR],
        capture_output=True,
        text=True,
    )
    print((done.stdout + done.stderr).replace(token or "\0", "***")[-800:])
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", PUBLIC_URL])
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"])
!apt-get -qq install -y libraqm0 fonts-noto-core > /dev/null
%pip install -q -e "/content/MangaTR[gpu,ocr]" pytest

In [ ]:
# 3) الموديلات (لو نزلت قبل كده هيتخطى التحميل)
%cd /content/MangaTR
M = "/content/models"
!python -m mangatr.gpu_check --models-dir $M --runs 1 --cpu-runs 0 --out /content/r.json | tail -3
!ls -la $M

In [ ]:
# 4) مفاتيح الترجمة من أسرار Colab (بتروح لمتغيرات البيئة، ما بتتطبعش)
import os

from google.colab import userdata

for name in ("GEMINI_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        os.environ[name] = userdata.get(name)
        print(name, "موجود")
    except Exception:
        print(name, "مش موجود")

In [ ]:
# 5) الصفحات: ارفعها يدوياً، أو zip من Drive (غيّر SOURCE)
import os
import zipfile

os.makedirs("/content/pages", exist_ok=True)
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    SOURCE = "/content/drive/MyDrive/chapter.zip"
    with zipfile.ZipFile(SOURCE) as z:
        z.extractall("/content/pages")
else:
    from google.colab import files

    for name, data in files.upload().items():
        with open(f"/content/pages/{name}", "wb") as handle:
            handle.write(data)
print(len(os.listdir("/content/pages")), "عنصر")

In [ ]:
# 6) التشغيل. غيّر TO (ar / en) والمزوّد (gemini / openai / anthropic).
# LIMIT=3 لتجربة سريعة قبل الفصل كامل (0 = كله)
TO, PROVIDER, LIMIT = "ar", "gemini", 3
%cd /content/MangaTR
IO = "--images /content/pages --models-dir /content/models --out /content/out"
OPTS = f"--to {TO} --provider {PROVIDER} --limit {LIMIT} --format zip"
!python -m mangatr.cli translate $IO $OPTS

In [ ]:
# 7) اعرض أول 3 صفحات مترجمة (مصغّرة) وتقرير الأخطاء
import glob
import json

from IPython.display import Image, display
from PIL import Image as PILImage

for path in sorted(glob.glob("/content/out/*.png"))[:3]:
    im = PILImage.open(path)
    im.thumbnail((900, 2400))
    small = path.replace(".png", "_small.jpg")
    im.convert("RGB").save(small, quality=85)
    display(Image(small))
with open("/content/out/report.json", encoding="utf-8") as handle:
    report = json.load(handle)
for p in report["pages"]:
    print(
        p["name"],
        p["blocks"],
        "نص،",
        p["translated"],
        "مترجم،",
        p["error"] or "",
        p["failed_blocks"],
        p["seconds"],
    )

In [ ]:
# 8) ملخص صغير للإرسال: report.json + أول 4 صفحات مصغّرة (بدل zip الفصل الكبير)
import glob
import zipfile

from google.colab import files
from PIL import Image as PILImage

with zipfile.ZipFile("/content/summary.zip", "w") as z:
    z.write("/content/out/report.json", "report.json")
    for i, path in enumerate(sorted(glob.glob("/content/out/*.png"))[:4]):
        im = PILImage.open(path)
        im.thumbnail((900, 2400))
        im.convert("RGB").save(f"/content/t{i}.jpg", quality=82)
        z.write(f"/content/t{i}.jpg", f"page{i + 1}.jpg")
files.download("/content/summary.zip")
# الفصل نفسه: /content/out/chapter.zip

## ابعتلي إيه

1. `summary.zip` (التقرير + أول صفحات مترجمة مصغّرة).
2. أي صفحة شكلها غلط (نص طالع برا الحبابة، تنظيف ناقص، ترجمة غريبة): ابعتها مع رقمها.
3. لو خلية فشلت، آخر 15 سطر من خطأها (من غير أي مفتاح).